In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import logging

from pathlib import Path
import anndata2ri
from bokeh.models import TabPanel, Tabs, ColorBar
from bokeh.plotting import show, output_file
from scipy.stats import median_abs_deviation
import seaborn as sns
import scrublet as scr
import seaborn as sns
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
from utils import interactive_embedding

%matplotlib inline

anndata2ri.activate()
%load_ext rpy2.ipython

rcb.logger.setLevel(logging.ERROR)

sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False, figsize=(8, 5))

pd.set_option("display.max_columns", None)
sc.settings.verbosity = 1

In [ ]:
BATCH = 1
PT = "PT-214"
side = "L"
FILTERED_RAW = "filtered"

In [ ]:
if BATCH == 1:
    batch_path = Path("../data/narsad_cellRanger_outs/")
else:
    batch_path = Path()

adata = sc.read_10x_mtx(
    batch_path / "blood" / f"{PT}-blood-{side}" / f"{FILTERED_RAW}_feature_bc_matrix"
)

adata

In [ ]:
FILTERED_RAW = "raw"
adata_raw = sc.read_10x_mtx(
    batch_path / "blood" / f"{PT}-blood-{side}" / f"{FILTERED_RAW}_feature_bc_matrix"
)
adata_raw

In [ ]:
adata.var_names_make_unique()
adata_raw.var_names_make_unique()

### 1) Highest expressed genes

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20)

#### Remove MALAT1

In [ ]:
malat1 = adata.var_names.str.startswith("MALAT1")
keep = np.invert(malat1)
adata = adata[:, keep]

print(adata.n_obs, adata.n_vars)

### 2) Filter low quality cells

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
adata

### 3) Low quality reads

In [ ]:
# mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes.
adata.var["hb"] = adata.var_names.str.contains(("^HB[^(P)]"))

In [ ]:
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True
)
adata

In [ ]:
# p1 = sns.displot(adata.obs["total_counts"], bins=100, kde=False)
# #sc.pl.violin(adata, 'total_counts')
# p2 = sc.pl.violin(adata, "pct_counts_mt")
# p3 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

In [ ]:
sc.pp.log1p(adata)
sc.tl.pca(adata, svd_solver="arpack", n_comps=50, use_highly_variable=False)
n_pcs = len(adata.uns["pca"]["variance_ratio"])
print(
    f"Explained variance of first {n_pcs} PCs = ",
    adata.uns["pca"]["variance_ratio"].sum(),
)

sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(
    adata,
    color=[
        "log1p_n_genes_by_counts",
        "log1p_total_counts",
        "pct_counts_mt",
        "pct_counts_ribo",
        "pct_counts_hb",
    ],
)

In [ ]:
def is_outlier(adata, metric: str, nmads: int):
    M = adata.obs[metric]
    outlier = (M < np.median(M) - nmads * median_abs_deviation(M)) | (
        np.median(M) + nmads * median_abs_deviation(M) < M
    )
    return outlier

In [ ]:
adata.obs["outlier"] = (
    is_outlier(adata, "log1p_total_counts", 5)
    | is_outlier(adata, "log1p_n_genes_by_counts", 5)
    | is_outlier(adata, "pct_counts_in_top_20_genes", 5)
)
adata.obs.outlier.value_counts()

In [ ]:
adata.obs["mt_outlier"] = is_outlier(adata, "pct_counts_mt", 3) | (
    adata.obs["pct_counts_mt"] > 8
)
adata.obs.mt_outlier.value_counts()

In [ ]:
print(f"Total number of cells: {adata.n_obs}")
adata = adata[(~adata.obs.outlier) & (~adata.obs.mt_outlier)].copy()

print(f"Number of cells after filtering of low quality cells: {adata.n_obs}")

In [ ]:
# p1 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")
# sc.pl.highest_expr_genes(adata, n_top=20)

In [ ]:
from utils import interactive_embedding

In [ ]:
# sc.pp.normalize_per_cell(adata)
# sc.pp.log1p(adata) # R equivalent of NormalizeData

sc.tl.pca(adata, svd_solver="arpack")
n_pcs = len(adata.uns["pca"]["variance_ratio"])
print(
    f"Explained variance of first {n_pcs} PCs = ",
    adata.uns["pca"]["variance_ratio"].sum(),
)
sc.pp.neighbors(adata, n_neighbors=20, n_pcs=50)
sc.tl.umap(adata)
# sc.pl.umap(adata, color=["MALAT1", "CD74"])

### 4) Soup removal and doublet detection

In [ ]:
adata_pp = adata.copy()
sc.pp.normalize_per_cell(adata_pp)
sc.pp.log1p(adata_pp)

In [ ]:
sc.pp.pca(adata_pp)
sc.pp.neighbors(adata_pp)
sc.tl.leiden(adata_pp, key_added="soupx_groups")

# Preprocess variables for SoupX
soupx_groups = adata_pp.obs["soupx_groups"]

In [ ]:
del adata_pp

cells = adata.obs_names
genes = adata.var_names
data = adata.X.T

adata_raw.var_names_make_unique()

malat1 = adata_raw.var_names.str.startswith("MALAT1")
keep = np.invert(malat1)
adata_raw = adata_raw[:, keep]
data_tod = adata_raw.X.T
del adata_raw

In [ ]:
%%R -i data -i data_tod -i genes -i cells -i soupx_groups -o out 
library(SoupX)
# specify row and column names of data
rownames(data) = genes
colnames(data) = cells
# ensure correct sparse format for table of counts and table of droplets
data <- as(data, "sparseMatrix")
data_tod <- as(data_tod, "sparseMatrix")

# Generate SoupChannel Object for SoupX 
sc = SoupChannel(data_tod, data, calcSoupProfile = FALSE)

# Add extra meta data to the SoupChannel object
soupProf = data.frame(row.names = rownames(data), est = rowSums(data)/sum(data), counts = rowSums(data))
sc = setSoupProfile(sc, soupProf)
# Set cluster information in SoupChannel
sc = setClusters(sc, soupx_groups)

# Estimate contamination fraction
sc  = autoEstCont(sc, doPlot=FALSE)
# Infer corrected table of counts and rount to integer
out = adjustCounts(sc, roundToInt = TRUE)

In [ ]:
adata.layers["counts"] = adata.X
adata.layers["soupX_counts"] = out.T
adata.X = adata.layers["soupX_counts"]

In [ ]:
print(f"Total number of genes: {adata.n_vars}")

# Min 3 cells - filters out 0 count genes
sc.pp.filter_genes(adata, min_cells=3)
print(f"Number of genes after cell filter: {adata.n_vars}")

#### Doublet with `scDblFinder`

In [ ]:
%%R
library(Seurat)

library(scater)
library(scDblFinder)
library(BiocParallel)

In [ ]:
data_mat = adata.X.T.copy()

In [ ]:
%%R -i data_mat -o doublet_score -o doublet_class

set.seed(123)
sce = scDblFinder(
    SingleCellExperiment(
        list(counts=data_mat),
    ) 
)
doublet_score = sce$scDblFinder.score
doublet_class = sce$scDblFinder.class

In [ ]:
adata.obs["scDblFinder_score"] = doublet_score
adata.obs["scDblFinder_class"] = doublet_class
adata.obs["scDblFinder_class"] = adata.obs["scDblFinder_class"].astype("object")

adata.obs.scDblFinder_class.value_counts()

#### Doubles with scrublet

In [ ]:
scrub = scr.Scrublet(adata.X, expected_doublet_rate=0.076)
(
    adata.obs["doublet_scores_scrublet"],
    adata.obs["predicted_doublets_scrublet"],
) = scrub.scrub_doublets(
    min_counts=2, min_cells=3, min_gene_variability_pctl=85, n_prin_comps=30
)

In [ ]:
# scrub.plot_histogram()

In [ ]:
adata.obs["predicted_doublets_scrublet"].value_counts()

In [ ]:
# adata.obs['predicted_doublets_scrublet'] =adata.obs['predicted_doublets_scrublet'].astype('str')

### 5) Normalization

#### log1p transform


In [ ]:
scales_counts = sc.pp.normalize_total(adata, target_sum=None, inplace=False)
# log1p transform
adata.layers["log1p_norm"] = sc.pp.log1p(scales_counts["X"], copy=True)

In [ ]:
# import matplotlib.pyplot as plt
# fig, axes = plt.subplots(1, 2, figsize=(10, 5))
# p1 = sns.histplot(adata.obs["total_counts"], bins=100, kde=False, ax=axes[0])
# axes[0].set_title("Total counts")
# p2 = sns.histplot(adata.layers["log1p_norm"].sum(1), bins=100, kde=False, ax=axes[1])
# axes[1].set_title("Shifted logarithm")
# plt.show()

#### scran

In [ ]:
from scipy.sparse import csr_matrix, issparse

In [ ]:
%%R
library(scran)
library(BiocParallel)

In [ ]:
# Preliminary clustering for differentiated normalisation
adata_pp = adata.copy()
sc.pp.normalize_total(adata_pp)
sc.pp.log1p(adata_pp)
sc.pp.pca(adata_pp, n_comps=15)
sc.pp.neighbors(adata_pp)
sc.tl.leiden(adata_pp, key_added="groups")

In [ ]:
data_mat = adata_pp.X.T
# convert to CSC if possible. See https://github.com/MarioniLab/scran/issues/70
if issparse(data_mat):
    if data_mat.nnz > 2**31 - 1:
        data_mat = data_mat.tocoo()
    else:
        data_mat = data_mat.tocsc()
ro.globalenv["data_mat"] = data_mat
ro.globalenv["input_groups"] = adata_pp.obs["groups"]

In [ ]:
del adata_pp

In [ ]:
%%R -o size_factors

size_factors = sizeFactors(
    computeSumFactors(
        SingleCellExperiment(
            list(counts=data_mat)), 
            clusters = input_groups,
            min.mean = 0.1,
            BPPARAM = MulticoreParam()
    )
)

In [ ]:
adata.obs["size_factors"] = size_factors
scran = adata.X / adata.obs["size_factors"].values[:, None]
adata.layers["scran_normalization"] = csr_matrix(sc.pp.log1p(scran))

In [ ]:
# import matplotlib.pyplot as plt
# fig, axes = plt.subplots(1, 2, figsize=(10, 5))
# p1 = sns.histplot(adata.obs["total_counts"], bins=100, kde=False, ax=axes[0])
# axes[0].set_title("Total counts")
# p2 = sns.histplot(
#     adata.layers["scran_normalization"].sum(1), bins=100, kde=False, ax=axes[1]
# )
# axes[1].set_title("log1p with Scran estimated size factors")
# plt.show()

#### Analytic Pearson residual

In [ ]:
# from scipy.sparse import csr_matrix, issparse

# analytic_pearson = sc.experimental.pp.normalize_pearson_residuals(adata, inplace=False)
# adata.layers["analytic_pearson_residuals"] = csr_matrix(analytic_pearson["X"])

In [ ]:
# fig, axes = plt.subplots(1, 2, figsize=(10, 5))
# p1 = sns.histplot(adata.obs["total_counts"], bins=100, kde=False, ax=axes[0])
# axes[0].set_title("Total counts")
# p2 = sns.histplot(
#     adata.layers["analytic_pearson_residuals"].sum(1), bins=100, kde=False, ax=axes[1]
# )
# axes[1].set_title("Analytic Pearson residuals")
# plt.show()

In [ ]:
# sc.experimental.pp.recipe_pearson_residuals(adata)

### sctransform v2

In [ ]:
# %%R -i adata
# # install sctransform >= 0.3.3
# # invoke sctransform - requires Seurat>=4.1
# seu <- CreateSeuratObject(adata, assay="RNA")
# adata <- SCTransform(adata, vst.flavor = "v2")

### 6) Feature selection

#### Deviance

In [ ]:
%%R
library(scry)


In [ ]:
import rpy2.robjects as ro

adata_deviance = adata.copy()
ro.globalenv["adata_deviance"] = adata_deviance

In [ ]:
adata_deviance.X

In [ ]:
%%R 
sce = devianceFeatureSelection(adata_deviance, assay="X")

In [ ]:
labels = [
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
    "pct_counts_ribo",
    "scDblFinder_score",
    "scDblFinder_class",
    "doublet_scores_scrublet",
    "predicted_doublets_scrublet",
]

In [ ]:
adata

In [ ]:
ro.r("rowData(sce)")

In [ ]:
binomial_deviance = ro.r("rowData(sce)$binomial_deviance").T
idx = binomial_deviance.argsort()[-3000:]
mask = np.zeros(adata_deviance.var_names.shape, dtype=bool)
mask[idx] = True

adata_deviance.var["highly_deviant"] = mask
adata_deviance.var["binomial_deviance"] = binomial_deviance
adata_deviance.X = adata.layers["log1p_norm"]

adata_deviance.var["highly_variable"] = adata_deviance.var["highly_deviant"]

adata_deviance.obs["predicted_doublets_scrublet"] = adata_deviance.obs[
    "predicted_doublets_scrublet"
].astype("str")

sc.pp.highly_variable_genes(adata, layer="scran_normalization")
sc.tl.pca(adata_deviance, svd_solver="arpack", n_comps=50, use_highly_variable=True)


sc.pp.neighbors(adata_deviance)
sc.tl.umap(adata_deviance)
sc.pl.umap(adata_deviance, color=labels, show=False, return_fig=False)

In [ ]:
adata_deviance

#### Seurat highly variables

In [ ]:
# adata_seurat = adata.copy()
# adata_seurat.X = adata_seurat.layers["log1p_norm"]

# sc.pp.normalize_total(adata_seurat)
# sc.pp.log1p(adata_seurat)

# sc.pp.highly_variable_genes(adata_seurat, n_top_genes=3000)
# sc.tl.pca(adata_seurat, svd_solver="arpack", n_comps=50, use_highly_variable=True)
# n_pcs = len(adata_seurat.uns["pca"]['variance_ratio'])
# print(f'Explained variance of first {n_pcs} PCs = ', adata_seurat.uns["pca"]['variance_ratio'].sum())

# sc.pp.neighbors(adata_seurat)
# sc.tl.umap(adata_seurat)
# # sc.pl.umap(adata_seurat, color=labels, show=False, return_fig=False)

#### Seurat v3 highly variables

In [ ]:
# adata_seuratv3 = adata.copy()
# sc.pp.normalize_total(adata_seuratv3)

# sc.pp.highly_variable_genes(adata_seuratv3, flavor='seurat_v3', n_top_genes=3000)
# sc.pp.log1p(adata_seuratv3)

# sc.tl.pca(adata_seuratv3, svd_solver="arpack", n_comps=50, use_highly_variable=True)
# print(f'Explained variance of first {n_pcs} PCs = ', adata_seuratv3.uns["pca"]['variance_ratio'].sum())

# sc.pp.neighbors(adata_seuratv3)
# sc.tl.umap(adata_seuratv3)
# # sc.pl.umap(adata_seuratv3, color=labels, show=False, return_fig=False)

#### pearson residuals highly variables

In [ ]:
# adata_pearson = adata.copy()
# # adata_pearson.X = adata_pearson.layers["analytic_pearson_residuals"]
# sc.pp.normalize_total(adata_pearson)
# sc.pp.log1p(adata_pearson)

# sc.experimental.pp.highly_variable_genes(
#         adata_pearson, flavor="pearson_residuals", n_top_genes=3000
#     )

# sc.tl.pca(adata_pearson, svd_solver="arpack", n_comps=50, use_highly_variable=True)
# print(f'Explained variance of first {n_pcs} PCs = ', adata_pearson.uns["pca"]['variance_ratio'].sum())

# sc.pp.neighbors(adata_pearson)
# sc.tl.umap(adata_pearson)
# # sc.pl.umap(adata_pearson, color=labels, show=False, return_fig=False)

In [ ]:
adata_deviance.obs["scDblFinder_class"] = adata_deviance.obs[
    "scDblFinder_class"
].astype("str")
adata_deviance.obs["predicted_doublets_scrublet"] = adata_deviance.obs[
    "predicted_doublets_scrublet"
].astype("str")

In [ ]:
# sc.tl.pca(adata, svd_solver="arpack", n_comps=50, use_highly_variable=False)
# n_pcs = len(adata.uns["pca"]['variance_ratio'])
# print(f'Explained variance of first {n_pcs} PCs = ', adata.uns["pca"]['variance_ratio'].sum())

# sc.pp.neighbors(adata)
# sc.tl.umap(adata)
# sc.pl.umap(adata, color=labels, show=False, return_fig=False)

In [ ]:
labels = [
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
    "pct_counts_ribo",
    "scDblFinder_score",
    "scDblFinder_class",
    "doublet_scores_scrublet",
    "predicted_doublets_scrublet",
]

In [ ]:
tabs = []

for label in labels:
    tabs.append(
        TabPanel(child=interactive_embedding(adata_deviance, label), title=label)
    )


# tabs.append(TabPanel(child=interactive_embedding(adata_deviance, "total_counts"), title="Deviance FS"))
# tabs.append(TabPanel(child=interactive_embedding(adata_seurat, "total_counts"), title="Seurat FS"))
# tabs.append(TabPanel(child=interactive_embedding(adata_seuratv3, "total_counts"), title="Seurat3 FS"))
# tabs.append(TabPanel(child=interactive_embedding(adata_pearson, "total_counts"), title="Pearson FS"))

p = Tabs(tabs=tabs)
output_file(f"../QC_plots/{PT}-blood-{side}_QC.html")
#
show(p)

In [ ]:
adata.write(f"../data/blood/{PT}-blood-{side}_QC_normalization.h5ad")

In [ ]:
del adata_deviance

In [ ]:
%%R
rm(adata_deviance)
gc()